In [1]:
import importlib.util
import sys
from pathlib import Path

SCHEMAS_DIR = Path("/home/hello/Projects/Statements/code/moltie/schemas")
assert SCHEMAS_DIR.exists(), f"Missing schemas dir: {SCHEMAS_DIR}"

PKG = "schemas"  # pretend package name

# Create an empty package module object for `schemas`
if PKG not in sys.modules:
    pkg_spec = importlib.util.spec_from_loader(PKG, loader=None)
    pkg_mod = importlib.util.module_from_spec(pkg_spec)
    pkg_mod.__path__ = [str(SCHEMAS_DIR)]  # mark as package
    sys.modules[PKG] = pkg_mod

def import_as_pkg(mod_name: str, file_path: Path):
    full_name = f"{PKG}.{mod_name}"
    spec = importlib.util.spec_from_file_location(full_name, str(file_path))
    assert spec and spec.loader, f"Cannot load spec for {file_path}"
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = mod
    spec.loader.exec_module(mod)
    return mod

query_object  = import_as_pkg("query_object",  SCHEMAS_DIR / "query_object.py")
verdict       = import_as_pkg("verdict",       SCHEMAS_DIR / "verdict.py")
negative_exit = import_as_pkg("negative_exit", SCHEMAS_DIR / "negative_exit.py")
run_config    = import_as_pkg("run_config",    SCHEMAS_DIR / "run_config.py")

QueryObject  = query_object.QueryObject
AtomQuery    = query_object.AtomQuery
Verdict      = verdict.Verdict
Anchor       = verdict.Anchor
NegativeExit = negative_exit.NegativeExit
RunConfig    = run_config.RunConfig

print("✅ Loaded schemas as an in-memory package namespace")


✅ Loaded schemas as an in-memory package namespace


In [2]:
from pathlib import Path
import PyPDF2

appeal_dir = Path("/media/hello/Vault/Tribunals/EAT_Appeals/")
appeal_name = "Z_v_A_UKEAT_0203_13_SM_.pdf" # <-- change this

assert appeal_dir.exists(), f"Missing dir: {appeal_dir}"


PDF_PATH = appeal_dir / appeal_name  # <-- change this
assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"

def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for i, page in enumerate(reader.pages):
            txt = page.extract_text() or ""
            out.append(txt)
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:1200])


PDF chars: 52945
Preview:
  Copyright 2013  Appeal No.  UKEAT /0203/13/SM  
    & UKEAT/0380/13/SM  
 
 
EMPLOYMENT APPEAL TRIBUNAL  
FLEETBANK HOUSE, 2 -6 SALISBURY SQUARE, LONDON, EC4Y 8JX  
 
 
 At the Tribunal  
 on 12th November 2013  
    Judgment handed down o n 9th December 2013  
 
 
Before  
THE HONOURABLE MR JUSTI CE LANGSTAFF  (PRESIDENT)  
MR I EZEKIEL  
MR H SINGH  
 
  
UKEAT/0203/13/SM  
 
Z  APPELLANT  
 
 
 
 
 
A RESPONDENT  
 
  
UKEAT/0380/13/SM  
 
A APPELLANT  
 
 
 
 
 
Z 
 RESPONDENT  
 
 
 
JUDGMENT  
 
 
UKEAT/0203/13/SM  
UKEAT/0308/13/SM  
   
 
 
 
 
 
 
 APPEARANCES  
 
 
 
 
 
For the Appellant  
(in UKEAT/0203/13/SM  
and the Respondent in  
UKEAT/0380/13/SM)  
 MR ANDREW WATSON  
(Representative)  
Instructed by:  
Free Representation Unit  
Ground Floor  
60 Gray's Inn Road  
London  
WC1X 8LU  
 
For the Respondent  
(in UKEAT/0203/13/SM  
and the Appellant in  
UKEAT/0380/13/SM)  
 MR BRUCE GARDINER  
 (of Counsel)  
Instructed by:  
Legal Services

In [3]:
# === Moltie LLM verifier: single-document smoke test (TRIMMED) ===
# End-to-end: PDF -> paras(with stable para_id) -> AtomQuery -> prompt -> Ollama -> Verdict

from pathlib import Path
import sys
import PyPDF2

# -----------------------
# 0) Paths / imports
# -----------------------
PROJECT_ROOT = Path("/home/hello/Projects/Statements/code")  # adjust if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from moltie.schemas.query_object import build_atom_from_y_path
from moltie.schemas.run_config import RunConfig
from moltie.llm.verifier_prompt import build_verifier_prompt
from moltie.llm.client import verify_with_ollama, LLMClientConfig

# -----------------------
# 1) Load one PDF -> text
# -----------------------
appeal_dir = Path("/media/hello/Vault/Tribunals/EAT_Appeals/")
appeal_name = "Z_v_A_UKEAT_0203_13_SM_.pdf"  # <-- change this
PDF_PATH = appeal_dir / appeal_name

assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"

def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            out.append(page.extract_text() or "")
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:800])

# -----------------------
# 2) Paragraph split + stable IDs (NO regex)
# -----------------------
def clean_para(s: str) -> str:
    # normalize whitespace without regex
    return " ".join((s or "").strip().split())

def to_paras(text: str, min_len: int = 40):
    paras = []
    buf = []
    n = 0

    for line in (text or "").splitlines():
        if line.strip():
            buf.append(line)
        else:
            if buf:
                p = clean_para("\n".join(buf))
                buf = []
                if len(p) >= min_len:
                    n += 1
                    paras.append({"para_id": f"p{n:05d}", "text": p})

    # flush tail
    if buf:
        p = clean_para("\n".join(buf))
        if len(p) >= min_len:
            n += 1
            paras.append({"para_id": f"p{n:05d}", "text": p})

    return paras

paras = to_paras(doc_text)
print("Paras:", len(paras))
assert paras, "No paragraphs extracted — PDF text extraction likely failed."

para_id_to_idx = {p["para_id"]: i for i, p in enumerate(paras)}

# -----------------------
# 3) AtomQuery (debug)
# -----------------------
Y_PATH = "/home/hello/Projects/Statements/output/Y_inferred.json"
Y_DEDUP_OUT = "/home/hello/Projects/Statements/output/Y_inferred_v2.dedup_view.json"

# comes from whatever step generates your evidence hits (debug stub here)
evidence_to_x = {
    "p00029": ["X1", "X3"],
    "p00030": ["X1"],
    "p00031": ["X4", "X1"],
}

atom, deduped_y = build_atom_from_y_path(
    atom_id="X_DEBUG",
    y_path=Y_PATH,
    evidence_to_x=evidence_to_x,
    dedup_out_path=Y_DEDUP_OUT,
)

print("AtomQuery x_tests (deduped):", atom.x_tests)
print("Wrote deduped Y view to:", Y_DEDUP_OUT)

cfg = RunConfig(
    anchors_required=2,
    thresh_score=70,
    thresh_conf=75,
    y_path=Y_PATH,
    y_dedup_out=Y_DEDUP_OUT,
)

# -----------------------
# 4) Evidence pack (deterministic, NO keyword heuristics)
#    Strategy:
#      - If evidence_to_x contains para_ids present in this doc, take those + neighbor window
#      - Else take a simple first-K slice (or mid-doc slice)
# -----------------------
K_DEBUG_PARAS = 12
NEIGHBOR_WINDOW = 3  # +/- around referenced para_ids

def select_debug_paras(paras, para_id_to_idx, preferred_para_ids):
    idxs = set()

    # include referenced paras + neighbors (when present)
    for pid in preferred_para_ids:
        i = para_id_to_idx.get(pid)
        if i is None:
            continue
        lo = max(0, i - NEIGHBOR_WINDOW)
        hi = min(len(paras) - 1, i + NEIGHBOR_WINDOW)
        idxs.update(range(lo, hi + 1))

    if idxs:
        chosen = [paras[i] for i in sorted(idxs)]
        # cap to K by taking a centered window around the median chosen index
        if len(chosen) > K_DEBUG_PARAS:
            mid = len(chosen) // 2
            half = K_DEBUG_PARAS // 2
            chosen = chosen[max(0, mid - half) : max(0, mid - half) + K_DEBUG_PARAS]
        return chosen, {"method": "evidence_window", "note": "para_id hits + neighbors"}

    # fallback: first K paras (or mid slice if doc is huge)
    if len(paras) <= K_DEBUG_PARAS:
        return paras, {"method": "first_k", "note": "doc shorter than K"}
    return paras[:K_DEBUG_PARAS], {"method": "first_k", "note": "no para_id hits; first-K fallback"}

preferred_para_ids = list(evidence_to_x.keys())
top_paras, retrieval_meta = select_debug_paras(paras, para_id_to_idx, preferred_para_ids)

evidence_pack = {
    "doc_id": PDF_PATH.stem,
    "doc_meta": {"source_path": str(PDF_PATH)},
    "paras": top_paras,
    "retrieval": {"method": retrieval_meta["method"], "note": retrieval_meta["note"], "score": None},
}

print("Evidence pack paras:", len(evidence_pack["paras"]))
print("Retrieval:", evidence_pack["retrieval"])
print("\n--- Evidence pack preview (para_id, snippet) ---")
for p in evidence_pack["paras"][:8]:
    print(p["para_id"], "|", p["text"][:180], "..." if len(p["text"]) > 180 else "")

# -----------------------
# 5) Build prompt + call Ollama
# -----------------------
prompt = build_verifier_prompt(atom=atom, evidence_pack=evidence_pack, cfg=cfg)

client_cfg = LLMClientConfig(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=800,
    max_retries=2,
)

verdict = verify_with_ollama(prompt, client_cfg)

# -----------------------
# 6) Inspect result
# -----------------------
print("\n=== VERDICT ===")
print(verdict.to_dict())

print("\n=== ANCHORS (verbatim) ===")
for a in verdict.anchors:
    print(f"- {a.para_id}: {a.quote[:180]}{'...' if len(a.quote)>180 else ''}")
    print(f"  why: {a.why_it_matters}")


PDF chars: 52945
Preview:
  Copyright 2013  Appeal No.  UKEAT /0203/13/SM  
    & UKEAT/0380/13/SM  
 
 
EMPLOYMENT APPEAL TRIBUNAL  
FLEETBANK HOUSE, 2 -6 SALISBURY SQUARE, LONDON, EC4Y 8JX  
 
 
 At the Tribunal  
 on 12th November 2013  
    Judgment handed down o n 9th December 2013  
 
 
Before  
THE HONOURABLE MR JUSTI CE LANGSTAFF  (PRESIDENT)  
MR I EZEKIEL  
MR H SINGH  
 
  
UKEAT/0203/13/SM  
 
Z  APPELLANT  
 
 
 
 
 
A RESPONDENT  
 
  
UKEAT/0380/13/SM  
 
A APPELLANT  
 
 
 
 
 
Z 
 RESPONDENT  
 
 
 
JUDGMENT  
 
 
UKEAT/0203/13/SM  
UKEAT/0308/13/SM  
   
 
 
 
 
 
 
 APPEARANCES  
 
 
 
 
 
For the Appellant  
(in UKEAT/0203/13/SM  
and the Respondent in  
UKEAT/0380/13/SM)  
 MR ANDREW WATSON  
(Representative)  
Instructed by:  
Free Representation Unit  
Ground Floor  
60 Gray's Inn Road  
Lond
Paras: 88
AtomQuery x_tests (deduped): ['X1']
Wrote deduped Y view to: /home/hello/Projects/Statements/output/Y_inferred_v2.dedup_view.json
Evidence pack paras: 9
Retrieval

### Moltie agent execution (single document)

This cell is the **minimal execution entry point** for Moltie.

It invokes the Moltie agent loop on:
- one document (identified by `doc_id`)
- pre-split paragraphs with stable `para_id`s
- a fully constructed `AtomQuery`
- runtime and LLM configuration

The agent loop evaluates the evidence and returns exactly one outcome:
- a **Verdict** if the atom is supported, or
- a **NegativeExit** if the agent cannot validate it.

This cell assumes all inputs are valid and integrated.
It represents the **canonical way Moltie is run**.


# Moltie — Single‑Document Flow

This file explains **what runs and in what order** when Moltie evaluates **one document against one atom**.

---

## Entry point

You call **one function only**:

```python
res = run_agent_on_one_doc(doc_id, paras, atom, run_cfg, llm_cfg)
```

The result is **exactly one** of:
- `Verdict` → accepted evidence‑anchored result  
- `NegativeExit` → explicit, audited stop condition

---

## What happens inside

1) **Retrieve (recall only)**  
`retrieve_windowed_evidence`  
Selects a small window of paragraphs and tracks coverage.

2) **Ask (build prompt)**  
`build_verifier_prompt`  
Compiles AtomQuery + evidence + strict Verdict JSON contract.

3) **Verify (LLM single‑shot)**  
`verify_with_ollama`  
Calls the LLM once and returns a schema‑valid `Verdict` (repair/retry if needed).

4) **Truth gate**  
Agent checks every anchor:
- `para_id` exists  
- quote is verbatim  
Failure ⇒ reject.

5) **Score progress**  
Agent computes a quality metric to detect improvement or plateau.

6) **Decide**  
- Accept ⇒ return `Verdict`  
- Plateau / exhausted ⇒ return `NegativeExit`  
- Else ⇒ refine query and loop

---

## Key rule

**The agent controls the loop.  
The LLM answers one question.**


In [4]:
from moltie.agent.loop import run_agent_on_one_doc
res = run_agent_on_one_doc(PDF_PATH.stem, paras, atom, cfg, client_cfg)
res.verdict.to_dict() if res.verdict else res.negative_exit.to_dict()

[moltie.loop] start doc_id='Z_v_A_UKEAT_0203_13_SM_' atom_id='X_DEBUG' n_paras=88
[moltie.loop] iter=1 rel=True score=0 conf=0 anchors=2 quality=46
[moltie.loop] QUOTE: 'The Tribunal concluded that the reason for dismissal was that there had been an accusation of historic child abuse against the Claimant. The matter was not a trivial one, but whether it was a sufficient reason depended on the circumstances.'
[moltie.loop] PARA : '15. The Tribunal concluded at paragraphs 6.5 and 6.6 that the reason for dismissal was th at there had been an accusation of historic child abuse against the Claimant. It asked whether the accusation was a substantial reason justifying the dismissal of a person holding the position that A held (the
[moltie.loop] INVALID_ANCHORS iter=2 bad=[('non_verbatim', 'p00028')] -> continuing
[moltie.loop] iter=3 rel=False score=0 conf=0 anchors=0 quality=0
[moltie.loop] PLATEAU iter=3 but continuing for coverage (plateau=2 visited_windows=7/8)
[moltie.loop] EXIT exhauste

{'atom_id': 'X_DEBUG',
 'reason': 'exhausted',
 'best_attempt': {'atom_id': 'string',
  'doc_id': 'string',
  'relevant': True,
  'matched_X': [],
  'precedent_score': 0,
  'confidence': 0,
  'anchors': [{'para_id': 'p00028',
    'quote': 'A bare accusation by itself, even of something so serious, cannot, in my finding, amount by itself to a substantial reason justifying a dismissal.',
    'why_it_matters': 'Highlights the legal standard for dismissals based on accusations.'},
   {'para_id': 'p00030',
    'quote': 'the school regarded the matter as substantial because if the allegation were true, and even if it were not, the fact that it had been made, meant that they could not run the risk of having the Claimant as caretaker.',
    'why_it_matters': 'Shows the practical impact of accusations on employment decisions.'}],
  'use_mode': 'support',
  'proposition_winner': 'unclear',
  'appeal_outcome': 'unknown',
  'successful_party': 'unclear',
  'distinguishers': [],
  'note': 'Discusse

In [5]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/hello/Projects/Statements/code")  # adjust if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import moltie.llm.verifier_prompt as vp
import moltie.llm.client as cl
import moltie.agent.loop as lp

importlib.reload(vp)
importlib.reload(cl)
importlib.reload(lp)

print("✅ reloaded moltie modules")


✅ reloaded moltie modules
